# Zero-Shot Pre-Labeling for Negative and Positive Reviews

This notebook uses a zero-shot classification model to pre-label PickMe reviews.

We create two separate labeled datasets:

1. Negative reviews → complaint categories
2. Positive reviews → satisfaction categories

These pre-labeled datasets will later be manually reviewed and corrected.

In [16]:
import pandas as pd
from pathlib import Path
from transformers import pipeline
from tqdm import tqdm
import torch
import re

In [22]:
RAW_PATH = Path("../data/raw/pickme_reviews_with_sentiment.csv")
PROCESSED_DIR = Path("../data/processed")

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

Load, Clean, and Filter Noise

In [24]:
# Load data
df = pd.read_csv(RAW_PATH)

# Keep English reviews only
df_en = df[df["language"].astype(str).str.strip().str.lower().isin(["english", "en"])].copy()

# Clean review text
df_en["review_text"] = df_en["review_text"].astype(str).str.strip()

# Remove empty reviews
df_en = df_en[df_en["review_text"].str.len() > 0].copy()

# Remove very short reviews
# Short reviews usually do not contain enough information for topic classification
df_en = df_en[df_en["review_text"].str.len() >= 15].copy()

# Clean sentiment labels safely
df_en["sentiment_clean"] = (
    df_en["sentiment"]
    .astype(str)
    .str.strip()
    .str.lower()
    .map(
        lambda x: "NEGATIVE" if "neg" in x
        else "POSITIVE" if "pos" in x
        else "OTHER"
    )
)

# Keep only Negative and Positive
# This removes "Not Analyzed"
df_en = df_en[df_en["sentiment_clean"].isin(["NEGATIVE", "POSITIVE"])].copy()

# Split into negative and positive
df_neg = df_en[df_en["sentiment_clean"] == "NEGATIVE"].copy().reset_index(drop=True)
df_pos = df_en[df_en["sentiment_clean"] == "POSITIVE"].copy().reset_index(drop=True)

print(f"Negative reviews after cleaning: {len(df_neg)}")
print(f"Positive reviews after cleaning: {len(df_pos)}")

Negative reviews after cleaning: 745
Positive reviews after cleaning: 349


Define category labels

In [7]:
NEGATIVE_LABELS = [
    "App bugs and technical issues",
    "Pricing, surge, and bidding",
    "Food delivery delays and missing items",
    "Driver behavior and safety",
    "Customer support and refunds",
    "Privacy and security concerns",
    "Other or unclear complaint"
]

POSITIVE_LABELS = [
    "App is easy to use and works well",
    "Pricing is fair and transparent",
    "Delivery was fast and reliable",
    "Driver was polite, professional, and safe",
    "Customer support was helpful",
    "Privacy and account security is good",
    "Other or unclear praise"
]

Load the zero-shot model

In [9]:
print("Loading model...")
classifier = pipeline(
    "zero-shot-classification", 
    model="facebook/bart-large-mnli", 
    device=0
)

Loading model...


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

Classification function

In [11]:
def pre_label_reviews(df, labels, batch_size=8):
    """
    This function applies a classifier to the review texts in a DataFrame.
    It batches the texts and applies the classifier to each batch.
    The predicted labels and confidence scores are then added as new columns
    to the DataFrame.

    Args:
        df (pandas.DataFrame): The DataFrame with review texts.
        labels (list): The list of labels to predict.
        batch_size (int, optional): The number of texts per batch. Defaults to 8.

    Returns:
        pandas.DataFrame: The DataFrame with predicted labels and confidence scores.
    """

    # Create a copy of the DataFrame
    df = df.copy()
    
    # Get the review texts
    texts = df["review_text"].tolist()

    # Initialize lists for predicted labels and confidence scores
    predicted_labels = []
    confidence_scores = []

    # Iterate over the texts in batches
    for i in tqdm(range(0, len(texts), batch_size)):
        # Get the current batch of texts
        batch_texts = texts[i : i + batch_size]

        # Apply the classifier to the batch of texts
        batch_results = classifier(batch_texts, labels)

        # Extract the predicted label and confidence score from each result
        for result in batch_results:
            predicted_labels.append(result["labels"][0])
            confidence_scores.append(result["scores"][0])

    # Add the predicted labels and confidence scores as new columns to the DataFrame
    df["predicted_category"] = predicted_labels
    df["category_confidence"] = confidence_scores

    # Return the updated DataFrame
    return df

Pre-label negative reviews

In [12]:
df_neg_labeled = pre_label_reviews(df_neg, NEGATIVE_LABELS, batch_size=8)

  0%|          | 0/106 [00:00<?, ?it/s]

100%|██████████| 106/106 [20:37<00:00, 11.67s/it]


Pre-label positive reviews

In [13]:
df_pos_labeled = pre_label_reviews(df_pos, POSITIVE_LABELS, batch_size=8)

100%|██████████| 75/75 [20:16<00:00, 16.23s/it]


Check quick results

In [14]:
print("Negative review category counts:")
print(df_neg_labeled["predicted_category"].value_counts())

print("\nPositive review category counts:")
print(df_pos_labeled["predicted_category"].value_counts())

Negative review category counts:
predicted_category
Other or unclear complaint                664
Driver behavior and safety                 71
Food delivery delays and missing items     42
App bugs and technical issues              33
Privacy and security concerns              16
Pricing, surge, and bidding                11
Customer support and refunds               10
Name: count, dtype: int64

Positive review category counts:
predicted_category
App is easy to use and works well            224
Other or unclear praise                      202
Customer support was helpful                  81
Driver was polite, professional, and safe     48
Delivery was fast and reliable                23
Pricing is fair and transparent                8
Privacy and account security is good           7
Name: count, dtype: int64


In [15]:
pd.set_option('display.max_colwidth', 150)

def inspect_other(df, other_label, name):
    df = df.copy()
    df['review_length'] = df['review_text'].str.len()

    other_df = df[df['predicted_category'] == other_label].copy()

    print(f"\n{name} 'Other' reviews: {len(other_df)}")

    print("\nReview length stats:")
    print(other_df['review_length'].describe())

    print("\nRandom examples:")
    sample_size = min(10, len(other_df))
    print(other_df.sample(sample_size, random_state=42)[
        ['review_text', 'predicted_category', 'category_confidence', 'review_length']
    ])

inspect_other(
    df_neg_labeled,
    other_label="Other or unclear complaint",
    name="Negative"
)

inspect_other(
    df_pos_labeled,
    other_label="Other or unclear praise",
    name="Positive"
)


Negative 'Other' reviews: 664

Review length stats:
count    664.000000
mean     133.454819
std      140.363387
min        3.000000
25%       26.000000
50%       77.500000
75%      196.000000
max      500.000000
Name: review_length, dtype: float64

Random examples:
                                                                                                                                               review_text  \
353  Today I had a very disappointing experience with your service. I have been using this app for the past 8 years, and I have never faced an incident...   
359                                                                                                                                        such a slow app   
606  Getting offers from drivers is probably the stupidest feature in the history of ride-sharing apps. Of course they're all going to offer the highes...   
285                                                                                                  